# Trace-Bench M1 — Minimal API Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/guru-code-expert/Trace-Bench/blob/m1/deliverable/notebooks/01_m1_minimal_api.ipynb)

This notebook validates the **M1 contracts**: canonical artifacts, deterministic IDs, and minimal runnable coverage across benches.

**Mode policy**: defaults to **real** (uses API key if present). If no key is found, falls back to **stub** with a clear warning and STUB label on outputs.

## Expected Outputs

- A new `runs/<run_id>/` folder with `meta/` + `jobs/` layout.
- `meta/config.snapshot.yaml`, `meta/manifest.json`, `meta/env.json` exist.
- `results.csv` contains `status` values (`ok`/`failed`/`skipped`).
- Internal non-trainable job shows `status=failed` with reason.
- If running in **real** mode, artifacts show `mode=real` and LLM4AD task produces a score.
- **2x2 matrix smoke**: `results.csv` with exactly 4 rows from 2 tasks x 2 trainers x 1 seed.

In [ ]:
# Mount Drive (optional) + compute persistent runs_dir + detect API key
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


def bench_dir(project="bench", sub="trace_bench", local="/content/bench"):
    drive_root = Path("/content/drive/MyDrive")
    root = drive_root if drive_root.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)

# --- Auto-detect API key (real mode by default) ---
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass

MODEL = os.environ.get("OPENROUTER_MODEL", "openrouter/openai/gpt-4o-mini")

if API_KEY:
    os.environ["OPENROUTER_API_KEY"] = API_KEY
    # Compatibility for OpenAI-style clients used internally by optimizers.
    os.environ["OPENAI_API_KEY"] = API_KEY
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
    MODE = "real"
    print(f"API key found - running in REAL mode (model: {MODEL})")
else:
    MODE = "stub"
    print("WARNING: No OPENROUTER_API_KEY found. Falling back to STUB mode.")
    print("         All outputs below are labeled STUB - not real LLM results.")

os.environ["TB_MODE"] = MODE
print(f"\nMode: {MODE}")

In [2]:
# Clone repos side-by-side (Trace-Bench + OpenTrace)
!git clone --depth 1 --branch m1/deliverable https://github.com/guru-code-expert/Trace-Bench.git
!git clone --depth 1 --branch experimental https://github.com/guru-code-expert/OpenTrace.git

%cd Trace-Bench

# System + Python deps
!apt-get update -y && apt-get install -y graphviz
!python -m pip install -U pip
!python -m pip install pyyaml pytest numpy matplotlib graphviz litellm==1.75.0


Cloning into 'Trace-Bench'...
remote: Enumerating objects: 315, done.
remote: Counting objects: 100% (315/315), done.
remote: Compressing objects: 100% (217/217), done.
remote: Total 315 (delta 42), reused 290 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (315/315), 3.86 MiB | 15.95 MiB/s, done.
Resolving deltas: 100% (42/42), done.
Cloning into 'OpenTrace'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 228 (delta 17), reused 115 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (228/228), 4.73 MiB | 28.34 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/Trace-Bench
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packa

In [3]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

echo "=== List trainers ==="
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench list-trainers

echo ""
echo "=== Validate config (strict) ==="
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench validate --config configs/m1_validation.yaml --strict --runs-dir "$RUNS_DIR"

echo ""
echo "=== Generate M1 run config (mode=$TB_MODE) ==="
cat > /content/m1_run.yaml <<YAML
runs_dir: runs
mode: $TB_MODE
seeds: [123]
max_workers: 1
fail_fast: false

tasks:
  - id: internal:code_param
  - id: internal:numeric_param
  - id: internal:multi_param
  - id: internal:non_trainable
  - id: trace_examples:greeting_stub
  - id: llm4ad:circle_packing
    eval_kwargs:
      timeout_seconds: 10
  - id: veribench:smoke_placeholder

trainers:
  - id: PrioritySearch
    params_variants:
      - threads: 2
        ps_steps: 1
        ps_batches: 1
        ps_candidates: 2
        ps_proposals: 2
        ps_mem_update: 1

  - id: GEPA-Base
    params_variants:
      - threads: 2
        gepa_iters: 1
        gepa_train_bs: 2
        gepa_merge_every: 2
        gepa_pareto_subset: 2
    optimizer: OPROv2
    optimizer_kwargs: {}

eval_kwargs:
  timeout_seconds: 10
YAML

echo "Config mode: $TB_MODE"
if [ "$TB_MODE" = "stub" ]; then
    echo "[STUB] Results below are from deterministic stub — not real LLM."
fi

echo ""
echo "=== Run M1 validation ==="
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config /content/m1_run.yaml --runs-dir "$RUNS_DIR"

=== List trainers ===
PrioritySearch	available
GEPA-Base	available
GEPA-UCB	available
GEPA-Beam	available

=== Validate config (strict) ===

=== Generate M1 run config (mode=real) ===
Config mode: real

=== Run M1 validation ===
PrioritySearch initialized with both short-term and long-term memory. Candidates will be merged into long-term memory every 1 iterations.
Epoch: 0. Iteration: 0
[Step 0] Test/test_score: 1.0
[Step 0] Algo/Average train score: 1.0
[Step 0] Update/n_iters: 0
[Step 0] Update/short_term_memory_size: 0
[Step 0] Update/long_term_memory_size: 2
[Step 0] Update/using_short_term_memory: False
[Step 0] Update/using_long_term_memory: True
[Step 0] Update/total_samples: 0
[Step 0] Update/best_candidate_priority: inf
[Step 0] Update/best_candidate_num_rollouts: 0
[Step 0] Update/num_exploration_candidates: 2
[Step 0] Update/exploration_candidates_mean_priority: inf
[Step 0] Update/exploration_candidates_average_num_rollouts: 0.0
[Step 0] Sample/mean_score: 1.0
[Step 0] Samp

usage: trace-bench [-h] {list-tasks,list-trainers,validate,run,ui} ...
trace-bench: error: unrecognized arguments: --runs-dir /content/drive/MyDrive/bench/2026-02-11/trace_bench
Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 6523.02it/s]
Backward: 100%|██████████| 2/2 [00:00<00:00, 7996.77it/s]
Calling optimizers: Generating 2 proposals for each of 2 batches: 100%|██████████| 4/4 [00:03<00:00,  1.19it/s]
Validating newly proposed candidates: Sampling 0 agents on 1 inputs: 0it [00:00, ?it/s]
Sampling training minibatch: Sampling 1 agents on 1 inputs: 100%|██████████| 1/1 [00:00<00:00, 4702.13it/s]
GEPA(base): child eval: 100%|██████████| 1/1 [00:00<00:00, 4485.89it/s]
Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 972.71it/s]
Backward: 100%|██████████| 2/2 [00:00<00:00, 819.84it/s]
Calling optimizers: Generating 2 proposals for each of 2 batches: 100%|██████████| 4/4 [00:10<00:00,  2.56s/it]


In [4]:
# Inspect latest run artifacts
import pathlib, json, pandas as pd

runs_root = pathlib.Path(RUNS_DIR)
candidates = sorted([p for p in runs_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)

run_dir = None
for p in reversed(candidates):
    if (p / "meta" / "config.snapshot.yaml").exists():
        run_dir = p
        break

if run_dir is None:
    for p in reversed(candidates):
        if (p / "config.snapshot.yaml").exists():
            run_dir = p
            break

if run_dir is None:
    raise FileNotFoundError("No run folder with config snapshot found under RUNS_DIR")

print("Run dir:", run_dir)

config_path = run_dir / "meta" / "config.snapshot.yaml"
env_path = run_dir / "meta" / "env.json"
manifest_path = run_dir / "meta" / "manifest.json"

if not config_path.exists():
    config_path = run_dir / "config.snapshot.yaml"
    env_path = run_dir / "env.json"

config_text = config_path.read_text()
print(config_text[:400])

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("Jobs in manifest:", len(manifest.get("jobs", [])))

df = pd.read_csv(run_dir / "results.csv")
df.head()


Run dir: /content/drive/MyDrive/bench/2026-02-11/trace_bench/20260211-104930-de435ae5
run_id: 20260211-104930-de435ae5
runs_dir: /content/drive/MyDrive/bench/2026-02-11/trace_bench
mode: real
seeds:
- 123
max_workers: 1
fail_fast: false
tasks:
- id: internal:code_param
  eval_kwargs:
    timeout_seconds: 10
- id: internal:numeric_param
  eval_kwargs:
    timeout_seconds: 10
- id: internal:multi_param
  eval_kwargs:
    timeout_seconds: 10
- id: internal:non_trainable
  eval_kwargs:
Jobs in manifest: 14


,run_id,job_id,task_id,suite,trainer_id,seed,status,score_initial,score_final,score_best,time_seconds,resolved_trainer_kwargs,resolved_optimizer_kwargs,eval_kwargs,feedback,tb_logdir
0,20260211-104930-de435ae5,6f3619dd9ae0,internal:code_param,internal,PrioritySearch,123,ok,1.0,1.0,1.0,7.705247,"{""memory_update_frequency"": 1, ""num_batches"": ...","{""memory_size"": 5, ""objective"": ""Match the tar...","{""timeout_seconds"": 10}",Correct,jobs/6f3619dd9ae0/tb
1,20260211-104930-de435ae5,c486ba93400f,internal:code_param,internal,GEPA-Base,123,ok,1.0,1.0,1.0,0.625392,"{""merge_every"": 2, ""num_iters"": 1, ""pareto_sub...","{""memory_size"": 5, ""objective"": ""Match the tar...","{""timeout_seconds"": 10}",Correct,jobs/c486ba93400f/tb
2,20260211-104930-de435ae5,778da61d2682,internal:numeric_param,internal,PrioritySearch,123,ok,-3.0,-0.0,-0.0,10.472214,"{""memory_update_frequency"": 1, ""num_batches"": ...","{""memory_size"": 5, ""objective"": ""Match the num...","{""timeout_seconds"": 10}",target=3.0,jobs/778da61d2682/tb
3,20260211-104930-de435ae5,4b3a7f322126,internal:numeric_param,internal,GEPA-Base,123,ok,-3.0,-0.0,-0.0,3.767528,"{""merge_every"": 2, ""num_iters"": 1, ""pareto_sub...","{""memory_size"": 5, ""objective"": ""Match the num...","{""timeout_seconds"": 10}",target=3.0,jobs/4b3a7f322126/tb
4,20260211-104930-de435ae5,0bfef35f6ef3,internal:multi_param,internal,PrioritySearch,123,ok,-1.0,-0.0,-0.0,4.724452,"{""memory_update_frequency"": 1, ""num_batches"": ...","{""memory_size"": 5, ""objective"": ""Make a+b matc...","{""timeout_seconds"": 10}",target=3.0,jobs/0bfef35f6ef3/tb


## 2x2 Bounded Matrix Smoke (Plan A+ Pareto)

Run exactly **2 tasks x 2 trainers x 1 seed = 4 jobs** and verify `results.csv` has 4 rows.

In [5]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

echo "=== 2x2 Matrix Smoke (mode=$TB_MODE) ==="

cat > /content/m1_matrix.yaml <<YAML
runs_dir: runs
mode: $TB_MODE
seeds: [123]
max_workers: 1
fail_fast: false

tasks:
  - id: internal:numeric_param
  - id: llm4ad:circle_packing
    eval_kwargs:
      timeout_seconds: 10

trainers:
  - id: PrioritySearch
    params_variants:
      - ps_steps: 1
        ps_batches: 1

  - id: GEPA-Base
    params_variants:
      - gepa_iters: 1
        gepa_train_bs: 2
        gepa_merge_every: 2
        gepa_pareto_subset: 2
YAML

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config /content/m1_matrix.yaml --runs-dir "$RUNS_DIR"

=== 2x2 Matrix Smoke (mode=real) ===
PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0
[Step 0] Test/test_score: -3.0
[Step 0] Algo/Average train score: -3.0
[Step 0] Update/n_iters: 0
[Step 0] Update/short_term_memory_size: 0
[Step 0] Update/long_term_memory_size: 2
[Step 0] Update/using_short_term_memory: False
[Step 0] Update/using_long_term_memory: True
[Step 0] Update/total_samples: 0
[Step 0] Update/best_candidate_priority: inf
[Step 0] Update/best_candidate_num_rollouts: 0
[Step 0] Update/num_exploration_candidates: 2
[Step 0] Update/exploration_candidates_mean_priority: inf
[Step 0] Update/exploration_candidates_average_num_rollouts: 0.0
[Step 0] Sample/mean_score: -3.0
[Step 0] Sample/num_samples: 2
[Step 0] Sample/self.n_epochs: 0
[Step 0] Algo/Number of training samples: 2
[Step 0] Parameter/__code0_copy:0: def emit(self, value):
        return value
[Step 0] Parameter/float:0: 0.0
Epoch: 0. Iteration: 1
[Step 1] Test/test_score: 0.0
[Step 1] Algo

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 4217.50it/s]
Backward: 100%|██████████| 2/2 [00:00<00:00, 1317.51it/s]
Calling optimizers: Generating 2 proposals for each of 2 batches: 100%|██████████| 4/4 [00:05<00:00,  1.28s/it]
Validating newly proposed candidates: Sampling 4 agents on 1 inputs: 100%|██████████| 4/4 [00:00<00:00, 2234.88it/s]
Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 2875.77it/s]
GEPA(base): child eval: 100%|██████████| 1/1 [00:00<00:00, 4888.47it/s]
Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 83.12it/s]
Backward: 100%|██████████| 2/2 [00:00<00:00, 7936.24it/s]
Calling optimizers: Generating 2 proposals for each of 2 batches: 100%|██████████| 4/4 [00:11<00:00,  2.96s/it]
Validating newly proposed candidates: Sampling 4 agents on 1 inputs: 100%|██████████| 4/4 [00:10<00:00,  2.50s/it]
Sampling training minibatch: Sampl

In [6]:
# Verify 2x2 matrix: exactly 4 rows in results.csv
import json, pathlib, pandas as pd

runs_root = pathlib.Path(RUNS_DIR)
candidates = sorted([p for p in runs_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)

matrix_dir = None
for p in reversed(candidates):
    summary_path = p / "summary.json"
    if not summary_path.exists():
        continue
    try:
        summary = json.loads(summary_path.read_text())
    except Exception:
        continue
    if summary.get("total_jobs") == 4:
        matrix_dir = p
        break

if matrix_dir is None:
    raise FileNotFoundError("No matrix run with total_jobs==4 found. Re-run the matrix cell.")

print("Matrix run dir:", matrix_dir)

df = pd.read_csv(matrix_dir / "results.csv")
print(f"\nresults.csv rows: {len(df)}  (expected: 4)")
assert len(df) == 4, f"Expected 4 rows, got {len(df)}"

summary = json.loads((matrix_dir / "summary.json").read_text())
print(f"summary.json: {summary}")
assert summary.get("total_jobs") == 4

print("\n--- Matrix results ---")
df[["task_id", "suite", "trainer_id", "seed", "status", "score_best"]]


Matrix run dir: /content/drive/MyDrive/bench/2026-02-11/trace_bench/20260211-105034-5e3554ca

results.csv rows: 4  (expected: 4)
summary.json: {'counts': {'ok': 4, 'failed': 0, 'skipped': 0}, 'total_jobs': 4}

--- Matrix results ---


,task_id,suite,trainer_id,seed,status,score_best
0,internal:numeric_param,internal,PrioritySearch,123,ok,-0.000000
1,internal:numeric_param,internal,GEPA-Base,123,ok,-0.000000
2,llm4ad:circle_packing,llm4ad,PrioritySearch,123,ok,0.789047
3,llm4ad:circle_packing,llm4ad,GEPA-Base,123,ok,0.840251
